## Convoluted Neural Network (CNN) - `keras`

All training code called here.

- Training Time Taken
- Test Accuracy

For Keras, the documentation is pretty rich, and gives good insight into how to use the api. This is a good starting reference. [https://keras.io/examples/vision/image_classification_from_scratch/](https://keras.io/examples/vision/image_classification_from_scratch/)

### Image Loading

There is difference in how you load images for linear classifiers as opposed to convoluted networks. I had spent time to understand the difference on loading images for `LinearSVC` and `MLPCLassifier` vs a Convolution Neural Network. For CNN training, it's assumed you have a large dataset, way more than what can be loaded in memory, to manage that, Keras supports options to load and classify data from a directory. Since have a small dataset, we can use it off memory.

#### `numpy`

A good amount of time was spent trying to make sure the data was available in a format which is "trainable" for the model. Reading a couple of chapters on `numpy.shape` and `numpy.reshape` helped with the transformation to prepare the data for training.


### Output Details

Each run writes these files, all with the same `RUN_ID`:

- `results/{RUN_ID}/03-custom-cnn/cnn_cv_{RUN_ID}.csv`: 5-fold cross-validation scores
- `results/{RUN_ID}/03-custom-cnn/cnn_cv_{RUN_ID}_fold{n}.csv`: per-epoch metrics for each fold
- `results/{RUN_ID}/03-custom-cnn/cnn_{model_name}.csv`: per-epoch metrics for the final fit
- `results/{RUN_ID}/03-custom-cnn/cnn_final_{RUN_ID}.csv`: fit-test score on the 8,000 test images
- `results/{RUN_ID}/03-custom-cnn/cnn_{model_name}.keras`: the final model


In [1]:
import datetime, base64, io, os, errno, math

import keras
import platform, sys

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from PIL import Image

from time import perf_counter
import tensorflow as tf

from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPool2D, AveragePooling2D
from keras.models import Sequential

from sklearn.model_selection import StratifiedKFold

from shared_library import (
    SEED,
    cross_validate_keras,
    fit_test,
    get_run_id,
    load_stl10_binary,
    results_dir,
)

In [2]:
RUN_ID = None

In [3]:
RUN_ID = get_run_id(RUN_ID)
RUN_DIR = results_dir(RUN_ID, "03-custom-cnn")

print(f"Architecture: {platform.machine()}")
print(f"Python Version: {sys.version.split()[0]}")
print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices("GPU")
print("GPUs found:", gpus)

Architecture: arm64
Python Version: 3.11.17
TensorFlow Version: 2.16.2
GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### Loading STL-10 Data


In [4]:
# 5000 images training dataset
# path to the binary train file with image data
TRAIN_DATA_PATH = "./stl-10/train_X.bin"
# path to the binary train file with labels
TRAIN_LABEL_PATH = "./stl-10/train_y.bin"
# 8000 images training dataset
# path to the binary test file with image data
TEST_DATA_PATH = "./stl-10/test_X.bin"
# path to the binary test file with labels
TEST_LABEL_PATH = "./stl-10/test_y.bin"

# path to the binary test file with labels
UNLABELED = "./stl-10/unlabeled_X.bin"

# STL-10 dimensions: 96x96 pixels, 3 channels (RGB)
HEIGHT = 96
WIDTH = 96
CHANNELS = 3

NUM_CLASSES = 10

# size of a single image in bytes
SIZE = HEIGHT * WIDTH * CHANNELS
INPUT_SHAPE = (HEIGHT, WIDTH, CHANNELS)


# Usage
X_train, y_train = load_stl10_binary(TRAIN_DATA_PATH, TRAIN_LABEL_PATH)
X_test, y_test = load_stl10_binary(TEST_DATA_PATH, TEST_LABEL_PATH)
# X_unlabeled = load_stl10_binary(UNLABELED)

print("Train shape:", X_train.shape)  # Output: (5000, 96, 96, 3)

# Rescaling
X_train = X_train.astype("float32") / 255
X_test = X_test.astype("float32") / 255


# One hot encoding labels, everything is mutable...
y_train = tf.keras.utils.to_categorical(y_train, num_classes=10)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=10)

print(y_train[:1])

Train shape: (5000, 96, 96, 3)
[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]]


### Training Levers

We can train the model by changing parameters listed below


In [ ]:
DENSE_LAYER_ACTIVATION = "softmax"
OBJECTIVE_FUNCTION = "categorical_crossentropy"

LEARNING_RATE = 1e-3

CNN_EPOCHS = 300
CNN_BATCH_SIZE = 64
CV_FOLDS = 5

FACTOR = 2
LAYERS = NUM_CLASSES * FACTOR

# Common accuracy metric for all outputs, but can use different metrics for different output
LOSS_METRICS = ["accuracy"]

N_JOBS = 5  # parallel processing of jobs

### Training a Regular CNN

[Keras Link](https://keras.io/getting_started/intro_to_keras_for_engineers/)

Using Sequential API


In [7]:
def build_cnn(lr):
    model = Sequential(
        [
            keras.Input(shape=INPUT_SHAPE),
            keras.layers.Conv2D(
                filters=32, kernel_size=(3, 3), padding="same", activation="relu"
            ),
            keras.layers.BatchNormalization(),
            keras.layers.Conv2D(
                filters=32, kernel_size=(3, 3), padding="same", activation="relu"
            ),
            keras.layers.BatchNormalization(),
            keras.layers.MaxPooling2D(pool_size=(30, 30)),
            keras.layers.Flatten(),
            keras.layers.Dropout(0.25),
            keras.layers.Dense(256, activation="relu"),
            keras.layers.Dropout(0.5),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(128, activation="relu"),
            keras.layers.Dropout(0.5),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dropout(0.5),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(NUM_CLASSES, activation="softmax"),
        ]
    )

    model.compile(
        loss=keras.losses.CategoricalCrossentropy(),
        optimizer=keras.optimizers.Adam(learning_rate=lr, weight_decay=1e-6),
        metrics=[keras.metrics.CategoricalAccuracy()],
    )
    return model


def modelling(density, enable_callbacks, lr, epochs, batch_size):

    model_name = f"relu_{density}_{lr}_{epochs}_{batch_size}_{RUN_ID}"
    output = f"{RUN_DIR}/cnn_{model_name}.csv"

    model = build_cnn(lr)

    callbacks = [
        keras.callbacks.CSVLogger(
            filename=output,
            separator=",",
        ),
    ]

    model.summary()

    history = model.fit(
        X_train,
        y_train,
        batch_size=batch_size,
        epochs=epochs,
        shuffle=True,
        validation_split=0.15,
        callbacks=callbacks,
        verbose=2,
    )
    model.summary()

    model.save(f"{RUN_DIR}/cnn_{model_name}.keras")

    print(history)

    return model

#### 5-fold Cross Validation


In [8]:
cnn_cv = cross_validate_keras(
    lambda: build_cnn(LEARNING_RATE),
    "cnn",
    X_train,
    y_train,
    epochs=CNN_EPOCHS,
    batch_size=CNN_BATCH_SIZE,
    run_id=RUN_ID,
    output_dir=RUN_DIR,
    n_splits=CV_FOLDS,
)

2026-11-02 22:57:54.267834: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-11-02 22:57:54.267870: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-11-02 22:57:54.267879: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
2026-11-02 22:57:54.267892: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-11-02 22:57:54.267902: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/300


2026-11-02 22:57:55.715080: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


63/63 - 15s - 245ms/step - categorical_accuracy: 0.1330 - loss: 2.6869 - val_categorical_accuracy: 0.1000 - val_loss: 2.8937
Epoch 2/300
63/63 - 9s - 137ms/step - categorical_accuracy: 0.1975 - loss: 2.2803 - val_categorical_accuracy: 0.1000 - val_loss: 3.4201
Epoch 3/300
63/63 - 8s - 127ms/step - categorical_accuracy: 0.2313 - loss: 2.0933 - val_categorical_accuracy: 0.1000 - val_loss: 3.1787
Epoch 4/300
63/63 - 9s - 145ms/step - categorical_accuracy: 0.2585 - loss: 1.9511 - val_categorical_accuracy: 0.1040 - val_loss: 2.9744
Epoch 5/300
63/63 - 9s - 148ms/step - categorical_accuracy: 0.2673 - loss: 1.8927 - val_categorical_accuracy: 0.1510 - val_loss: 2.6574
Epoch 6/300
63/63 - 9s - 150ms/step - categorical_accuracy: 0.2965 - loss: 1.8033 - val_categorical_accuracy: 0.1940 - val_loss: 2.4119
Epoch 7/300
63/63 - 10s - 156ms/step - categorical_accuracy: 0.3018 - loss: 1.7836 - val_categorical_accuracy: 0.2250 - val_loss: 2.1672
Epoch 8/300
63/63 - 9s - 150ms/step - categorical_accuracy

,fold,val_accuracy,val_loss,training_minutes
0,1,0.569,1.210742,43.293820
1,2,0.594,1.278276,34.615090
2,3,0.556,1.358723,33.941585
3,4,0.602,1.206985,33.918308
4,5,0.588,1.202477,58.546433


cnn: 0.5818 +/- 0.0189


#### Fit & Accuracy Test


In [9]:
start_time = perf_counter()
model = modelling(NUM_CLASSES, True, LEARNING_RATE, CNN_EPOCHS, CNN_BATCH_SIZE)
fit_test(
    model,
    "cnn",
    X_test,
    y_test,
    training_minutes=(perf_counter() - start_time) / 60,
    batch_size=CNN_BATCH_SIZE,
    run_id=RUN_ID,
    output_dir=RUN_DIR,
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 96, 96, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 96, 96, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 96, 96, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 96, 96, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 3, 3, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 288)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 288)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │        73,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 127,978 (499.91 KB)

 Trainable params: 126,954 (495.91 KB)

 Non-trainable params: 1,024 (4.00 KB)

Epoch 1/300
67/67 - 22s - 328ms/step - categorical_accuracy: 0.1402 - loss: 2.6956 - val_categorical_accuracy: 0.1093 - val_loss: 2.6479
Epoch 2/300
67/67 - 14s - 204ms/step - categorical_accuracy: 0.1941 - loss: 2.3049 - val_categorical_accuracy: 0.1093 - val_loss: 2.9646
Epoch 3/300
67/67 - 13s - 196ms/step - categorical_accuracy: 0.2271 - loss: 2.1229 - val_categorical_accuracy: 0.1093 - val_loss: 2.9348
Epoch 4/300
67/67 - 14s - 205ms/step - categorical_accuracy: 0.2459 - loss: 1.9787 - val_categorical_accuracy: 0.1107 - val_loss: 2.6847
Epoch 5/300
67/67 - 14s - 210ms/step - categorical_accuracy: 0.2628 - loss: 1.8938 - val_categorical_accuracy: 0.1880 - val_loss: 2.4017
Epoch 6/300
67/67 - 14s - 202ms/step - categorical_accuracy: 0.2767 - loss: 1.8223 - val_categorical_accuracy: 0.2173 - val_loss: 2.2126
Epoch 7/300
67/67 - 13s - 199ms/step - categorical_accuracy: 0.2974 - loss: 1.7828 - val_categorical_accuracy: 0.2240 - val_loss: 2.0395
Epoch 8/300
67/67 - 13s - 201ms/step - ca

... (3 outputs left out)

 Trainable params: 126,954 (495.91 KB)

 Non-trainable params: 1,024 (4.00 KB)

 Optimizer params: 253,910 (991.84 KB)

125/125 ━━━━━━━━━━━━━━━━━━━━ 25s 199ms/step - categorical_accuracy: 0.6011 - loss: 1.1582
cnn: test accuracy 0.6011, test loss 1.1582
